# Module 6 — Code Along: Data-Driven Automation (the bank-accounts story)

The capstone: a **data file** becomes **API actions**, with a **report** to show for it. We reuse the M4 model (`BankAccount`) and stand in for the M5 client with a `FakeAccountsAPI` — no live server. Files go to a temp dir.

Section by section, one code cell per beat — run top to bottom:

**Section 1** the pattern · **Section 2** validating the source · **Section 3** drive the API + report

## Section 1 — The pattern

Data-driven = the **data lives outside**, the **logic is fixed**. Every such job is the same shape: source → process each record → report.

In [ ]:
# 1.1 · The shape — source → process → report — is just one loop over records.
records = [{"id": 1, "owner": "Ada"}, {"id": 2, "owner": ""}]   # SOURCE (data, varies)
report = {"done": [], "failed": []}
for r in records:                          # PROCESS (logic, fixed): same steps per record
    if r["owner"]:
        report["done"].append(r["id"])
    else:
        report["failed"].append(r["id"])
print(report)                              # SINK: a report of what happened — {'done':[1],'failed':[2]}

## Section 2 — Validating the source

A CSV is **all strings**, so "validate" also means **coerce**. JSON already carries real types — but it still can't promise the **shape**: fields present, named right, within limits. Hand either one to the same Pydantic model: it coerces what needs coercing and checks everything, in one step. A bad record raises.

In [ ]:
# 2.1 · DictReader — one dict per row, every value a string.
import csv, json, tempfile, os
from pydantic import BaseModel, Field, ValidationError

class BankAccount(BaseModel):
    id: int = Field(ge=1)
    owner: str = Field(min_length=1)
    balance: float = Field(ge=0)

tmpdir = tempfile.mkdtemp()
csv_path = os.path.join(tmpdir, "accounts.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "owner", "balance"])
    w.writerow([1, "Ada", 1500.0])
    w.writerow([2, "Lin", 800.0])
    w.writerow([3, "", -5])                 # a deliberately bad row: empty owner, negative balance

for row in csv.DictReader(open(csv_path)):
    print(row)                              # all values are str: {'id':'1','balance':'1500.0',...}

In [ ]:
# 2.2 · model_validate coerces the strings AND checks the rules; a bad row raises.
good = BankAccount.model_validate({"id": "1", "owner": "Ada", "balance": "1500.0"})
print(type(good.balance).__name__, good.balance)            # float 1500.0  (coerced from "1500.0")
try:
    BankAccount.model_validate({"id": "3", "owner": "", "balance": "-5"})
except ValidationError as e:
    print("bad row ->", [er["type"] for er in e.errors()])  # ['string_too_short','greater_than_equal']

In [ ]:
# 2.3 · Same model, whatever the source — swap CSV for JSON and nothing changes.
json_path = os.path.join(tmpdir, "accounts.json")
json.dump([{"id": 1, "owner": "Ada", "balance": 1500.0},
           {"id": 2, "owner": "", "balance": -5}],            # 2nd record is bad
          open(json_path, "w"))
records = json.loads(open(json_path).read())
print(type(records[0]["balance"]).__name__)                   # float — JSON already typed it

# ...but 'already typed' is not 'valid'. Same model_validate, same gate:
print("ok:", BankAccount.model_validate(records[0]).owner)    # Ada
try:
    BankAccount.model_validate(records[1])                    # empty owner + negative balance
except ValidationError as e:
    print("bad shape ->", [er["type"] for er in e.errors()])
try:
    BankAccount.model_validate({"id": 5})                     # missing owner + balance
except ValidationError as e:
    print("missing  ->", [er["loc"][0] for er in e.errors()]) # ['owner', 'balance']

## Section 3 — Drive the API + report

One bad row must not kill the batch. Keep the two failure kinds apart — bad data (never sent) vs the server rejecting a well-formed row — and write a report of exactly what happened.

In [ ]:
# 3.1 · A FakeAccountsAPI stands in for the M5 client: POST creates; a duplicate id -> 409.
class APIError(Exception):
    def __init__(self, status):
        super().__init__(str(status)); self.status_code = status

class FakeAccountsAPI:
    def __init__(self): self._db = {}
    def create(self, account: dict):
        if account["id"] in self._db:
            raise APIError(409)              # server says no (duplicate)
        self._db[account["id"]] = account
        return account

api = FakeAccountsAPI()
api.create({"id": 1, "owner": "Ada", "balance": 1500.0})
try:
    api.create({"id": 1, "owner": "Ada", "balance": 1500.0})  # duplicate
except APIError as e:
    print("server rejected -> status", e.status_code)         # 409

In [ ]:
# 3.2 · The loop — validate each CSV row, send the good ones, sort failures into buckets.
api = FakeAccountsAPI()
api.create({"id": 1, "owner": "Existing", "balance": 100.0})  # id=1 already taken -> a duplicate below

created, validation_errors, api_errors = [], [], []
for n, row in enumerate(csv.DictReader(open(csv_path))):
    try:
        acct = BankAccount.model_validate(row)
    except ValidationError as e:
        validation_errors.append({"row": n, "errors": e.errors()})   # bad data — never sent
        continue
    try:
        api.create(acct.model_dump())
        created.append(acct.id)
    except APIError as e:
        api_errors.append({"row": n, "status": e.status_code})       # server said no

print("created:", created, "| bad data:", len(validation_errors), "| rejected:", len(api_errors))
# row0 id=1 -> 409 (taken) ; row1 id=2 -> created ; row2 bad -> validation_error

In [ ]:
# 3.3 · The report is the deliverable — write it to disk. Day 3 tests this exact shape.
report = {
    "summary": {"created": len(created),
                "validation_errors": len(validation_errors),
                "api_errors": len(api_errors)},
    "created": created,
    "validation_errors": validation_errors,
    "api_errors": api_errors,
}
report_path = os.path.join(tmpdir, "import_report.json")
json.dump(report, open(report_path, "w"), indent=2)
print(json.dumps(report["summary"]))      # {"created": 1, "validation_errors": 1, "api_errors": 1}

## Next — Day 3: test what we built

We now have the full Day-2 stack: a typed server, an `APIClient`, and a data-driven importer that turns a file into API calls and a **report**.

**Next we lock it down.** Day 3 is pytest: unit tests, mocking the client so tests need no server, parametrizing across status codes, coverage, and CI — and the **system-under-test is this `import_report.json` shape**.